<a href="https://colab.research.google.com/github/ibnu-ahmedin/afaan-oromo-hybrid-sentiment-analysis/blob/main/Hybrid_RuleBased_AfriBERTa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================================
# HYBRID Rule-based and Afriberta Model – LATE-FUSION
# ==========================================================
!pip install -U transformers datasets accelerate scikit-learn -q

import pandas as pd
import numpy as np
import torch
import re
import matplotlib.pyplot as plt

from datasets import Dataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback
)
from torch.nn.functional import softmax

# ==========================================================
# 1. LOAD DATA
# ==========================================================
train_df = pd.read_csv('/content/train.csv')
val_df   = pd.read_csv('/content/val.csv')
test_df  = pd.read_csv('/content/test.csv')
print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))

# ==========================================================
# 2. DATASETS + AfriBERTa
# ==========================================================
train_dataset = Dataset.from_pandas(train_df[["Text", "label"]])
val_dataset   = Dataset.from_pandas(val_df[["Text", "label"]])
test_dataset  = Dataset.from_pandas(test_df[["Text", "label"]])

model_name = "castorini/afriberta_base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

def tokenize_function(example):
    return tokenizer(example["Text"], padding="max_length", truncation=True, max_length=128)

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset   = val_dataset.map(tokenize_function, batched=True)
test_dataset  = test_dataset.map(tokenize_function, batched=True)

train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
val_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"accuracy": accuracy_score(labels, preds)}

training_args = TrainingArguments(
    output_dir="./afriberta_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=4,
    weight_decay=0.02,
    warmup_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=200,
    save_total_limit=2,
    seed=42
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)
trainer.train()

# AfriBERTa predictions
predictions = trainer.predict(test_dataset)
afri_probs = softmax(torch.tensor(predictions.predictions), dim=1).numpy()
afri_confs = np.max(afri_probs, axis=1)

# ==========================================================
# 3.RULE-BASED
# ==========================================================
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '@user', text)
    text = re.sub(r'[\U0001F600-\U0001FFFF]', '', text)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    text = re.sub(r"[^\w\s']", '', text)
    return ' '.join(text.split())

test_df["clean_text"] = test_df["Text"].apply(clean_text)

lexicon_df = pd.read_csv('/content/Final_Lexicon_v11_auto.csv')
lexicon_df['Word'] = lexicon_df['Word'].str.strip().str.lower()
lexicon_df['Category'] = lexicon_df['Category'].str.strip().str.lower()

polarity_dict = {}
intensifier_dict = {}
diminisher_dict = {}
negation_set = set()
phrase_dict = {}

for _, row in lexicon_df.iterrows():
    word = row['Word']
    score = row['Score']
    cat = row['Category']
    if cat in ['positive', 'negative']:
        polarity_dict[word] = score
    elif cat == 'intensifier':
        intensifier_dict[word] = score
    elif cat == 'diminisher':
        diminisher_dict[word] = score
    elif cat == 'negation':
        negation_set.add(word)
    if len(word.split()) > 1:
        phrase_dict[word] = score

def tokenize(text):
    if not isinstance(text, str):
        return []
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    words = re.findall(r"\b[\w']+\b", text)
    return words

def rule_based_sentiment(text):
    tokens = tokenize(text)
    if not tokens:
        return 1

    total_score = 0.0
    matched_words = set()
    text_joined = " ".join(tokens)

    # Phrase scoring
    for phrase, score in phrase_dict.items():
        if phrase in text_joined:
            total_score += score
            matched_words.update(phrase.split())

    # Word frequency
    word_counts = {}
    for w in tokens:
        word_counts[w] = word_counts.get(w, 0) + 1

    # Word-level scoring
    for i, word in enumerate(tokens):
        if word in matched_words:
            continue
        if word not in polarity_dict:
            continue

        score = polarity_dict[word]
        score *= min(1.5, word_counts[word])

        # Morphological negation rules
        if i > 0 and tokens[i-1] == "miti":
            score = -score
        elif i > 0 and tokens[i-1] in ["hin", "in"]:
            if word.endswith('u'):
                score = -score
        if i > 0 and (tokens[i-1] == "naan" or tokens[i-1].endswith('n')):
            if word.endswith(('e', 'u')):
                score = -score
        if word.endswith("ree") and word not in polarity_dict:
            score -= 0.5

        # Intensifier / Diminisher
        if i > 0:
            prev = tokens[i-1]
            if prev in intensifier_dict:
                score *= intensifier_dict[prev]
            elif prev in diminisher_dict:
                score *= diminisher_dict[prev]

        total_score += score

    # Contrast boost
    if any(w in {"garuu", "garu","immoo", "ammo", "ta'us", "malee"} for w in tokens):
        total_score *= 1.1

    norm_score = total_score / (len(tokens) + 1)

    if abs(norm_score) < 0.22:
        return 1
    elif norm_score > 0.25:
        return 2
    elif norm_score < -0.25:
        return 0
    else:
        return 1

# ==========================================================
# 4. ABLATION STUDY
# ==========================================================
alphas = [0.0, 0.20, 0.40, 0.60, 0.80, 1.0]
results = []
all_preds = {}
print("\nABLATION STUDY")

for alpha in alphas:
    preds = []
    for i in range(len(test_df)):
        rb_label = rule_based_sentiment(test_df["clean_text"].iloc[i])

        if alpha == 0.0:
            pred = rb_label
        else:
            afri_p = afri_probs[i]
            rb_probs = np.array([0.05, 0.15, 0.80] if rb_label == 2 else
                                [0.80, 0.15, 0.05] if rb_label == 0 else
                                [0.15, 0.70, 0.15])
            final = alpha * afri_p + (1 - alpha) * rb_probs
            pred = np.argmax(final)

        preds.append(pred)

    acc = accuracy_score(test_df["label"], preds)
    results.append(acc)
    all_preds[alpha] = preds
    print(f"Alpha {alpha}: {acc:.4f}")

best_idx = np.argmax(results)
best_alpha = alphas[best_idx]
best_preds = np.array(all_preds[best_alpha])

print(f"\n BEST ALPHA: {best_alpha}")
print(f"\n BEST HYBRID ACCURACY: {max(results):.4f}")

# ==========================================================
# 5. Classification Report + Confusion Matrix
# ==========================================================
print("\nClassification Report:\n")
print(classification_report(test_df["label"], best_preds,
                            target_names=["Negative","Neutral","Positive"]))

cm = confusion_matrix(test_df["label"], best_preds)
print("\nConfusion Matrix:\n", cm)
ConfusionMatrixDisplay(cm, display_labels=["Negative","Neutral","Positive"]).plot(cmap="Blues")
plt.title(f"Hybrid Confusion Matrix (Best α={best_alpha})")
plt.show()

# ==========================================================
# 6. Ablation Graph
# ==========================================================
plt.figure(figsize=(9,6))
plt.plot(alphas, results, marker='o', linewidth=2.5)
plt.title("Hybrid Ablation Curve")
plt.xlabel("Alpha (AfriBERTa Weight)")
plt.ylabel("Accuracy")
plt.grid(True)
plt.show()

# ==========================================================
# 7. Save Predictions
# ==========================================================
pd.DataFrame({
    "y_true": test_df["label"],
    "hybrid_preds": best_preds
}).to_csv("/content/hybrid_preds.csv", index=False)

print("\nBest predictions saved")